In [20]:
import modin.pandas as pd
import numpy as np
import gensim
from tqdm import tqdm 
import re
import os
import nltk
import ast
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.similarities import WmdSimilarity
from datetime import datetime

pd.set_option('display.max_colwidth', None)

In [5]:
def preprocess_text(text):
    text = nltk.word_tokenize(text)
    text=[word.lower() for word in text if word.isalpha()]
    filtered_text = [word for word in text if word not in stopwords.words('english') and not(re.match(r"\d",word))]
    lemmatized_text = [lemmatizer.lemmatize(word) for word in filtered_text]
    return lemmatized_text

In [8]:
jobs_data_dir = "../Job Data/"

In [9]:
dataframes = []
# Read all files in the directory
for file_name in os.listdir(jobs_data_dir):
    if '2024' in file_name: 
        print(file_name)
        file_path = os.path.join(jobs_data_dir, file_name)
        tmp_df = pd.read_csv(file_path, usecols = ['jobPostingId', 'title', 'description', 'tokens', 'gender_category'])
        tmp_df = tmp_df.dropna(how='any')
        dataframes.append(tmp_df)
job_postings_df = pd.concat(dataframes)
print(len(job_postings_df))
job_postings_df.head()

annotated_data_2024_batch_2_removed_words_tokenized.csv
annotated_data_2024_removed_words_tokenized.csv
77975


,jobPostingId,title,description,gender_category,tokens
0,3823301365,Python Full Stack Developer/Lead/Architect,"Skills- Python, Django, React/AngularExperience- 4+ yrsLocation- Hyderabad\nJob Description-\nâ¢ Python + Django, Angular/React Develop, and maintain robust, scalable backend systems using Python.â¢ Ensure technology solutions align with company architectural standards.â¢ Collaborate seamlessly with cross-functional teams for the integration of platform services.â¢ Foster close collaboration with frontend developers, ensuring a harmonious backend-frontend integration.â¢ Proactively identify and resolve issues promptly.â¢ Strong full-stack development skills in Python, including expertise in frameworks like Django (Expert).â¢ Experience with microservice architecture, end-to-end UI/API integration, and knowledge of API protocols like REST, gRPC, and GraphQL (Advanced).â¢ Knowledge of Caching technologies and DBMS technologies like MySQL, PostGres, MongoDB, and database schema design (Advanced).â¢ Strong problem-solving, communication, and organizational skills (Advanced).â¢ Proficiency in UI modern x like AngularJS or ReactJS (Intermediate).â¢ Proficient in drafting coding practices and designing highly scalable, secure, and maintainable software solutions (Intermediate).â¢ Experience in building large-scale platforms handling real-time complex transactions and troubleshooting distributed systems (Intermediate).",fem,"['python', 'full', 'stack', 'python', 'django', 'hyderabad', 'job', 'python', 'django', 'develop', 'maintain', 'robust', 'scalable', 'backend', 'system', 'using', 'ensure', 'technology', 'solution', 'align', 'company', 'architectural', 'collaborate', 'seamlessly', 'team', 'integration', 'platform', 'foster', 'close', 'collaboration', 'frontend', 'developer', 'ensuring', 'harmonious', 'proactively', 'identify', 'resolve', 'issue', 'strong', 'development', 'skill', 'python', 'including', 'expertise', 'framework', 'like', 'django', 'expert', 'experience', 'microservice', 'architecture', 'integration', 'knowledge', 'api', 'protocol', 'like', 'rest', 'grpc', 'graphql', 'advanced', 'knowledge', 'caching', 'technology', 'dbms', 'technology', 'like', 'mysql', 'postgres', 'mongodb', 'database', 'schema', 'design', 'advanced', 'strong', 'communication', 'organizational', 'skill', 'advanced', 'proficiency', 'ui', 'modern', 'x', 'like', 'angularjs', 'reactjs', 'intermediate', 'proficient', 'drafting', 'coding', 'practice', 'designing', 'highly', 'scalable', 'secure', 'maintainable', 'software', 'solution', 'intermediate', 'experience', 'building', 'platform', 'handling', 'complex', 'transaction', 'troubleshooting', 'distributed', 'system', 'intermediate']"
1,3823301077,Senior Data Engineer - w2 only,"Expertise:8+ years of relevant industry experience with a BS/MastersExperience with distributed processing technologies and frameworks, such as Hadoop, Spark, Kafka, and distributed storage systems (e.g., HDFS, S3)Demonstrated ability to analyze large data sets to identify gaps and inconsistencies, provide data insights, and advance effective product solutionsExpertise with ETL schedulers such as Apache Airflow, Luigi, Oozie, AWS Glue or similar frameworksSolid understanding of data warehousing concepts and hands-on experience with relational databases (e.g., PostgreSQL, MySQL) and columnar databases (e.g., Redshift, BigQuery, HBase, ClickHouse)Excellent written and verbal communication skillsA Typical Day:Design, build, and maintain robust and efficient data pipelines that collect, process, and store data from various sources, including user interactions, financial details, and external data feeds.Develop data models that enable the efficient analysis and manipulation of data for merchandising optimization. Ensure data quality, consistency, and accuracy.Build scalable data pipelines (SparkSQL & Scala) leveraging Airflow scheduler/executor frameworkCollaborate with cross-functional teams, including Data Scie

In [10]:
# Concatenate text for vectorization
job_postings_df['text'] = job_postings_df['title'] + ' ' + job_postings_df['description']

In [6]:
# # Load preprocessed datasets
# female_cvs_path = "User Data/female CVs/"
# female_cvs = []
# for file_name in os.listdir(female_cvs_path):
#     tmp_df = pd.read_csv(os.path.join(female_cvs_path, file_name)) 
#     female_cvs.append(tmp_df)
# female_cvs = pd.concat(female_cvs)
# female_cvs = female_cvs.head(25000).rename(columns = {'0': 'CV'})
# female_cvs['gender'] = 'female'
# print(len(female_cvs))
# female_cvs.head()

In [7]:
# # Load preprocessed datasets
# male_cvs_path = "User Data/male CVs"
# male_cvs = []
# for file_name in os.listdir(male_cvs_path):
#     tmp_df = pd.read_csv(os.path.join(male_cvs_path, file_name))
#     male_cvs.append(tmp_df)
# male_cvs = pd.concat(male_cvs).head(25000).rename(columns = {'0': 'CV'})
# male_cvs['gender'] = 'male'
# print(len(male_cvs))
# male_cvs.head()

In [8]:
# cvs_df = pd.concat([female_cvs, male_cvs])
# cvs_df.head()

In [3]:
cvs_df = pd.read_csv("../User Data/cvs_tokenized_final.csv")
cvs_df.head()

,CV,gender,text,tokens
0,"```plaintext\n**Summary** \nDynamic and detail-oriented software engineer with 12-14 years of progressive experience in developing robust applications using a variety of programming languages including TypeScript, SQL, and Java. Proven expertise in leveraging web and database technologies such as Node.js, ASP.NET Core, Firebase, and MongoDB. Highly adept in employing modern frameworks and tools to streamline development processes and enhance application performance. Committed to delivering innovative solutions and continuous improvement in the software development lifecycle.\n\n**Skills** \n- **Programming Languages:** TypeScript, SQL, Java \n- **Databases:** Firebase, MongoDB \n- **Web Frameworks:** Node.js, ASP.NET Core \n- **Other Frameworks:** .NET, Keras \n- **Tools:** Git, Ansible \n- **IDEs:** Sublime Text, IntelliJ \n- **Operating Systems:** Windows, macOS \n\n**Experience** \n- Developed and maintained complex software applications, leading projects from conception through deployment. \n- Collaborated with cross-functional teams to design and implement scalable solutions, enhancing user experience and operational efficiency. \n- Utilized TypeScript and SQL for backend services and database management, demonstrating fluency in data-driven application development. \n- Leveraged frameworks such as Node.js and ASP.NET Core to build responsive web applications, driving increased engagement and usability. \n- Implemented CI/CD practices using Git and Ansible, resulting in improved deployment times and reduced rollout errors. \n\n**Education** \nBachelorâ€™s Degree in Computer Science \n[Your University Name] \n[Year of Graduation] \nBrazil \n```",female,"```plaintext\n**Summary** \nDynamic and detail-oriented software engineer with 12-14 years of progressive experience in developing robust applications using a variety of programming languages including TypeScript, SQL, and Java. Proven expertise in leveraging web and database technologies such as Node.js, ASP.NET Core, Firebase, and MongoDB. Highly adept in employing modern frameworks and tools to streamline development processes and enhance application performance. Committed to delivering innovative solutions and continuous improvement in the software development lifecycle.\n\n**Skills** \n- **Programming Languages:** TypeScript, SQL, Java \n- **Databases:** Firebase, MongoDB \n- **Web Frameworks:** Node.js, ASP.NET Core \n- **Other Frameworks:** .NET, Keras \n- **Tools:** Git, Ansible \n- **IDEs:** Sublime Text, IntelliJ \n- **Operating Systems:** Windows, macOS \n\n**Experience** \n- Developed and maintained complex software applications, leading projects from conception through deployment. \n- Collaborated with cross-functional teams to design and implement scalable solutions, enhancing user experience and operational efficiency. \n- Utilized TypeScript and SQL for backend services and database management, demonstrating fluency in data-driven application development. \n- Leveraged frameworks such as Node.js and ASP.NET Core to build responsive web applications, driving increased engagement and usability. \n- Implemented CI/CD practices using Git and Ansible, resulting in improved deployment times and reduced rollout errors. \n\n**Education** \nBachelorâ€™s Degree in Computer Science \n[Your University Name] \n[Year of Graduation] \nBrazil \n```","['plaintext', 'summary', 'dynamic', 'software', 'engineer', 'year', 'progressive', 'experience', 'developing', 'robust', 'application', 'using', 'variety', 'programming', 'language', 'including', 'typescript', 'sql', 'java', 'proven', 'expertise', 'leveraging', 'web', 'database', 'technology', 'core', 'firebase', 'mongodb', 'highly', 'adept', 'employing', 'modern', 'framework', 'tool', 'streamline', 'development', 'process', 'enhance', 'application', 'performance', 'committed', 'delivering', 'innovative', 'solution', 'continuous', 'improvement', 'software', 'development', 'lifecycle', 'skill', 'program

In [11]:
# TF-IDF Vectorization
vectorizer = TfidfVectorizer()
job_tfidf = vectorizer.fit_transform(job_postings_df['text'])
cvs_tfidf = vectorizer.transform(cvs_df['text'])

In [12]:
# def recommend_tfidf(cv_index, top_n=10):
#     """Recommend jobs based on TF-IDF cosine similarity."""
#     similarities = cosine_similarity(cvs_tfidf[cv_index], job_tfidf).flatten()
#     top_indices = similarities.argsort()[-top_n:][::-1]
#     return job_postings_df.iloc[top_indices][['jobPostingId', 'title', 'description', 'gender_category']]

In [37]:
# Step 1: Load dictionary and extract keys
with open('Keyword Repository/fem_keywords_with_counts_removed.txt', 'r', encoding='utf-8') as f:
    fem_keyword_dict = ast.literal_eval(f.read())  # safely evaluate string as dictionary
FEM_WORDS = list(fem_keyword_dict.keys())

with open('Keyword Repository/masc_keywords_with_counts_removed.txt', 'r', encoding='utf-8') as f:
    masc_keyword_dict = ast.literal_eval(f.read())  # safely evaluate string as dictionary
MASC_WORDS = list(masc_keyword_dict.keys())

GENDERED_WORDS = set(MASC_WORDS + FEM_WORDS)
GENDERED_WORDS

{'a position with responsibility for',
 'able to express onself clearly verbally and writing',
 'able to make others enthusiastic',
 'able to motivate others',
 'achieve',
 'achievement',
 'achieves',
 'achieving',
 'active',
 'actively',
 'adventure',
 'adventures',
 'adventurous',
 'adventurously',
 'advice',
 'advocate',
 'affectionate',
 'aggress',
 'aggresses',
 'aggression',
 'aggressive',
 'aggressively',
 'ambassador',
 'ambition',
 'ambitious',
 'aspiration',
 'aspirational',
 'aspirations',
 'assert',
 'asserting',
 'assertings',
 'assertion',
 'assertive',
 'assertives',
 'asserts',
 'athlete',
 'athletic',
 'attentive',
 'autonomous',
 'autonomy',
 'boast',
 'boasting',
 'boasts',
 'business sense',
 'care',
 'challenge',
 'challenges',
 'challenging',
 'challengings',
 'cheer',
 'cheerfulness',
 'child',
 'childhood',
 'collaborate',
 'collaborating',
 'collaboration',
 'collaborative',
 'command',
 'commands',
 'commercial',
 'commercials',
 'commit',
 'commitment',
 'com

In [38]:
def recommend_tfidf(cv_index, top_n=10, alpha=0.01):
    """
    Recommend jobs based on TF-IDF cosine similarity,
    with a penalty for shared gendered words.
    
    Parameters:
    - cv_index: index of the CV in the TF-IDF matrix
    - top_n: number of top jobs to return
    - alpha: penalty per shared gendered word (tune this hyperparameter)
    """
    # Compute cosine similarities
    similarities = cosine_similarity(cvs_tfidf[cv_index], job_tfidf).flatten()

    # Get text of the CV
    cv_text = cvs_df.iloc[cv_index]['text'].lower().split()
    cv_words = set(cv_text)

    # Adjust similarity by penalizing shared gendered words
    adjusted_similarities = []
    for i, job_text in enumerate(job_postings_df['description']):
        job_words = set(job_text.lower().split())
        shared_gendered = GENDERED_WORDS.intersection(cv_words & job_words)
        penalty = alpha * len(shared_gendered)
        adjusted_score = similarities[i] - penalty
        adjusted_similarities.append(adjusted_score)

    # Get top N indices from adjusted scores
    adjusted_similarities = np.array(adjusted_similarities)
    top_indices = adjusted_similarities.argsort()[-top_n:][::-1]

    return job_postings_df.iloc[top_indices][['jobPostingId', 'title', 'description', 'gender_category']]

In [39]:
all_recommendations = []
for cv_index in tqdm(range(len(cvs_df))):
    recs = recommend_tfidf(cv_index)    
    recs = recs.reset_index(drop=True) 
    recs['rank'] = recs.index + 1         # Rank starts at 1
    recs['cv_index'] = cv_index           # Track which CV this is for
    recs['CV'] = cvs_df.iloc[cv_index]['CV']
    recs['applicant_gender'] = cvs_df.iloc[cv_index]['gender']
    all_recommendations.append(recs)    

  0%|▏                                                                           | 49/16000 [10:16<55:42:51, 12.57s/it]


KeyboardInterrupt: 

In [ ]:
recommendations_df = pd.concat(all_recommendations, ignore_index=True)
cols = [ 'cv_index', 'CV', 'applicant_gender', 'jobPostingId', 'title', 'description', 'gender_category', 'rank']
recommendations_df = recommendations_df[cols] 
recommendations_df.head()

In [ ]:
recommendations_df.to_csv(f'../Recommendation Data/tf_idf_{len(recommendations_df)}_penalty.csv')